# Python for Scientific Computing
## Sebastian Ohlmann, Klaus Reuter
## Max Planck Computing and Data Facility, Garching

# Using CUDA with Python

* pyCUDA
* CUDA kernel, wrap using Cython
* Numba

## pyCUDA
* [Documentation](https://sysbio.ioc.ee/projects/f2py2e/usersguide/index.html)
* Usage:
    * Write the CUDA kernel in a python string (might be parametrized)
    * The code is compiled and loaded to the GPU behind the scenes
    * Block and grid layout have to be specified
    * Arrays are copied back and forth
    * No cleanup needed

## pyCUDA example
* simple multiplication

In [ ]:
import pycuda.autoinit
import pycuda.driver as drv
import numpy

from pycuda.compiler import SourceModule
# define source module with CUDA code in a string
mod = SourceModule("""
__global__ void multiply_them(float *dest, float *a, float *b)
{
  const int i = threadIdx.x;
  dest[i] = a[i] * b[i];
}
""")
# extract function
multiply_them = mod.get_function("multiply_them")

a = numpy.random.randn(400).astype(numpy.float32)
b = numpy.random.randn(400).astype(numpy.float32)

dest = numpy.zeros_like(a)
# call the function
multiply_them(
        drv.Out(dest), drv.In(a), drv.In(b),
        block=(400,1,1), grid=(1,1))

print(dest-a*b)

## More advanced example: correlation function
* Given positions and velocities, compute autocorrelation of velocities depending on distance of particles
* Particles at irregular positions $\to$ quadratic complexity
* different versions in `pycuda/correlations`
    * 2 python versions
    * cython version
    * MPI version
    * pyCUDA version
* Parallel versions: need reduce at end

### python version

In [ ]:
def velcorrelation(pos, vel, real_edges, nbins):
    # return arrays
    velcorr = np.zeros((nbins,6), dtype=np.float64)
    numbin = np.zeros(nbins, dtype=np.int64)
    # loop over given cell indices
    for k in range(pos.shape[0]):
        # loop over particles with index smaller than k
        for j in range(k):
            dx = pos[j,0] - pos[k,0]
            dy = pos[j,1] - pos[k,1]
            dz = pos[j,2] - pos[k,2]

            dist = np.sqrt(dx*dx + dy*dy + dz*dz)
            
            ind = 0
            # loop over radial bins
            for i in range(nbins):
                if dist < real_edges[i+1]:
                  ind = i
                  break
            # compute velocity correlation
            numbin[ind] += 1
            velcorr[ind,0] += vel[k,0] * vel[j,0]
            velcorr[ind,1] += vel[k,1] * vel[j,1]
            velcorr[ind,2] += vel[k,2] * vel[j,2]
            velcorr[ind,3] += vel[k,0] * vel[j,1]
            velcorr[ind,4] += vel[k,0] * vel[j,2]
            velcorr[ind,5] += vel[k,1] * vel[j,2]
    return velcorr, numbin

### pyCUDA version
* python part:

In [ ]:
def velcorrelation_gpu(pos, vel, real_edges, nbins):
    npart = pos.shape[0]
    # generate code
    codefinal = code.substitute(npart="%d"%npart, nbins="%d"%nbins)
    mod = SourceModule(codefinal)
    func = mod.get_function("velcorrelation")

    # return arrays
    sumcorr = np.zeros((nbins,6), dtype=np.float64)
    numbin = np.zeros(nbins, dtype=np.int64)

    func(cuda.In(pos.astype(np.float64)), cuda.In(vel.astype(np.float64)), 
        cuda.In(real_edges.astype(np.float64)), 
        cuda.InOut(sumcorr), cuda.InOut(numbin),
        block=(512,1,1), grid=(4096,1))

    return sumcorr, numbin

### pyCUDA version
* array initialization:

In [ ]:
from string import Template
code = Template("""
    #include <stdio.h>
    #include <math.h>

    __global__ void velcorrelation(double *pos, double *vel,
          double* real_edges, double *velcorr, unsigned long long int *numbin) 
    {
      const int npart = ${npart};  // templated here!
      const int nbins = ${nbins};  // templated here!
      double dx, dy, dz, dist;
      int i, j, k;
      unsigned long long int numbin_cache[nbins];
      double velcorr_cache[nbins*6];

      for (k = 0; k < nbins; k++)
        {
          numbin_cache[k] = 0;
          velcorr_cache[k*6+0] = 0.0;
          velcorr_cache[k*6+1] = 0.0;
          velcorr_cache[k*6+2] = 0.0;
          velcorr_cache[k*6+3] = 0.0;
          velcorr_cache[k*6+4] = 0.0;
          velcorr_cache[k*6+5] = 0.0;
        }

### pyCUDA version
* main loop:

In [ ]:
""""
    // static distribution of indices to threads -> get from block and grid dimensions
    for (i = blockIdx.x * blockDim.x + threadIdx.x; 
         i < npart; 
         i += blockDim.x * gridDim.x) 
        {
          for (j = 0; j < i; j++)
            {
              dx = pos[j*3 + 0] - pos[i*3 + 0];
              dy = pos[j*3 + 1] - pos[i*3 + 1];
              dz = pos[j*3 + 2] - pos[i*3 + 2];

              dist = sqrt(dx*dx + dy*dy + dz*dz);
              // get bin index for dist
              for (k = 0; k < nbins; k++)
                {
                  if (dist < real_edges[k+1]) break;
                }

              numbin_cache[k] += 1;
              velcorr_cache[k*6+0] += vel[i*3 + 0] * vel[j*3 + 0];
              velcorr_cache[k*6+1] += vel[i*3 + 1] * vel[j*3 + 1];
              velcorr_cache[k*6+2] += vel[i*3 + 2] * vel[j*3 + 2];
              velcorr_cache[k*6+3] += vel[i*3 + 0] * vel[j*3 + 1];
              velcorr_cache[k*6+4] += vel[i*3 + 0] * vel[j*3 + 2];
              velcorr_cache[k*6+5] += vel[i*3 + 1] * vel[j*3 + 2];
            }
        }

### pyCUDA version
* reduction over threads:

In [ ]:
""""
      // Reducing velcorr and numbin
      for (k = 0; k < nbins; k++)
        {
          // reduce in block
          numbin_cache[k] = blockReduceSum(numbin_cache[k]);
          velcorr_cache[6*k + 0] = blockReduceSumf(velcorr_cache[6*k + 0]);
          velcorr_cache[6*k + 1] = blockReduceSumf(velcorr_cache[6*k + 1]);
          velcorr_cache[6*k + 2] = blockReduceSumf(velcorr_cache[6*k + 2]);
          velcorr_cache[6*k + 3] = blockReduceSumf(velcorr_cache[6*k + 3]);
          velcorr_cache[6*k + 4] = blockReduceSumf(velcorr_cache[6*k + 4]);
          velcorr_cache[6*k + 5] = blockReduceSumf(velcorr_cache[6*k + 5]);

          // reduce among blocks
          if (threadIdx.x == 0)
            {
              atomicAdd(&numbin[k], numbin_cache[k]);
              atomicAdd_d(&velcorr[6*k + 0], velcorr_cache[6*k + 0]);
              atomicAdd_d(&velcorr[6*k + 1], velcorr_cache[6*k + 1]);
              atomicAdd_d(&velcorr[6*k + 2], velcorr_cache[6*k + 2]);
              atomicAdd_d(&velcorr[6*k + 3], velcorr_cache[6*k + 3]);
              atomicAdd_d(&velcorr[6*k + 4], velcorr_cache[6*k + 4]);
              atomicAdd_d(&velcorr[6*k + 5], velcorr_cache[6*k + 5]);
            }
        }
    }
    """)

### pyCUDA version
* reduction in warp (smallest unit on GPU, executes one common instruction at a time):

In [ ]:
""""
    __inline__ __device__
    double warpReduceSumf(double val) {
      for (int mask = warpSize/2; mask > 0; mask /= 2) 
        val += __shfl_down(val, mask);
      return val;
    }

    __inline__ __device__
    double blockReduceSumf(double val) {
      static __shared__ double shared[32]; // Shared mem for 32 partial sums
      int lane = threadIdx.x % warpSize;
      int wid = threadIdx.x / warpSize;

      val = warpReduceSumf(val);     // Each warp performs partial reduction

      if (lane==0) shared[wid]=val; // Write reduced value to shared memory

      __syncthreads();              // Wait for all partial reductions

      // read from shared memory only if that warp existed
      val = (threadIdx.x < blockDim.x / warpSize) ? shared[lane] : 0;

      if (wid==0) val = warpReduceSumf(val); // Final reduce within first warp

      return val;
    }

""""

## Some timings

* Timings on a Tesla K20 (Kepler) for 1000 elements:

 gpu took  0.24s,
 cython took  0.030  s,
 python v1 took  5.94  s,
 python v2 took  8.28  s

* Timings on a Tesla K20 (Kepler), MPI on 10 cores (Ivy bridge) for 20000 elements:

 gpu took  0.25  s,
 cython took  23.18  s,
 mpi took  2.41  s

* Timings on a Tesla K20 (Kepler), MPI on 20 cores (1 Ivy bridge node) for 40000 elements:

 gpu took  0.30  s,
 mpi took  4.85  s
 
* Timings on a Tesla K20 (Kepler), MPI on 20 cores (1 Ivy bridge node) for 100000 elements:

 gpu took  0.8  s,
 mpi took  29.77  s
 
 
 ** $\to$ for some problems, GPUs are very effective! **


## CUDA kernel *plus* Cython interface
Usage:
* create a library (shared object, `.so`) of your CUDA device code (compute kernel) and the host function (memory allocation, transfer, kernel launch, transfer, deallocation), e.g.  
  `nvcc --shared -o libhello.so hello.cu --compiler-options '-fPIC'`
* write a Cython interface for the host function
* proceed as shown in `cython/c_interface_shared_object`  

Note for completeness:
* it is possible to compile both the CUDA code and the Cython interface into the same shared object (Python module), though not recommended

### Application: Cadishi
* joint project: MPI for Biophysics and MPCDF
* purpose: compute radial distribution functions from MD trajectories
* parallelization approach
    * Python multiprocessing
    * CPU C++ kernel, GPU CUDA kernel, Cython interface
    * HDF5 IO
* see `setup.py` for an example `nvcc` invocation
* https://github.com/bio-phys/cadishi
![Cadishi](fig/cadishi_histo2_combined_bins_prelim.svg)

## Numba

* Numba supports the implementation of CUDA kernels directly in Python
* fully transparent handling of NumPy arrays
* advanced features: explicit device and memory management possible, streams, atomics, etc.

Example below adapted from
https://numba.pydata.org/numba-doc/dev/cuda/kernels.html

In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def increment_by_one(x):
    i = cuda.grid(1)  # position in the 1D CUDA grid
    if i < x.size:  # check array boundaries
        x[i] += 1

x = np.zeros(1024*1024)        

threadsperblock = 32
blockspergrid = (x.size + (threadsperblock - 1)) // threadsperblock
increment_by_one[blockspergrid, threadsperblock](x)